## This script: morality_conditioned_pct inference (on Benchmark PVQ + PCT)

### generate morality_conditioned_pct prompts

In [ ]:
from morality.morality_prompt_utils_PVQ import all_morality_cases, morality_groups, morality_options, morality_topics
import pandas as pd
from politics.utils.pct_helpers import calculate_pct_coordinates, plot_pct
import numpy as np
import random
import torch

N_PER_SCORE = 1     
RNG_SEED = 42
MORAL_MODE = 2 
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

RAW_CSV = "./politic_morality/morality/data/morality_prompt_raw_PVQ.csv"
PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"

if MORAL_MODE == 2:
    OUT_CSV = f"./politic_morality/data/prompts/morality_binary_conditioned_pct_prompts_PVQ.csv"
else:
    OUT_CSV = f"./politic_morality/data/prompts/morality_{N_PER_SCORE}samples_conditioned_pct_prompts_PVQ.csv"

binary_map = {0: 0, 5: 1, "0": 0, "5": 1}
binary_text_map_agree = {
    0: "Not like me",
    1: "Like me",
}

PCT_COL_CANDIDATES = ["full_prompt"]


def parse_answer_id_text(cell):
    if isinstance(cell, str) and " sep|sep " in cell:
        sid, stext = cell.split(" sep|sep ", 1)
        try:
            return int(sid.strip()), stext.strip()
        except Exception:
            return None, cell
    return None, str(cell)


def build_prompts_from_raw(
    raw_csv=RAW_CSV,
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV,
    n_per_score=N_PER_SCORE
):
    df = pd.read_csv(raw_csv)
    df_pct = pd.read_csv(pct_csv)

    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")

    q_cols = sorted(
                    [c for c in df.columns if c.startswith("question ") and not c.endswith("id")],
                    key=lambda x: int(x.split()[-1])
                )
    a_cols = sorted([c for c in df.columns if c.startswith("answer ")], key=lambda x: int(x.split()[-1]))
    for i, acol in enumerate(a_cols, start=1):
        df[f"a{i}_id"] = df[acol].apply(lambda x: parse_answer_id_text(x)[0])
        df[f"a{i}_text"] = df[acol].apply(lambda x: parse_answer_id_text(x)[1])
    df["answer_sig"] = df[[f"a{i}_id" for i in range(1,7)]].apply(tuple, axis=1) # answer option list: (1,3,2,4,2,1,None,None...) 

    df_sorted = df.sort_values(["topic", "score"], ascending=[True, True])
    grouped = df_sorted.groupby(["topic", "score"], sort=False)

    out_rows = []
    for (topic, score), g in grouped: # key: (topic, score), dataFrame: g
        if MORAL_MODE == 2:
            if int(score) not in (3, -3):
                continue   
        
        g_nodup = g.drop_duplicates(subset=["answer_sig"]) # check if answer_sig is duplicate
        if len(g_nodup) > n_per_score:
            g_pick = g_nodup.sample(n=n_per_score, random_state=RNG_SEED)
            # one sample for score 3 and -3
        else:
            g_pick = g_nodup

        g_pick = g_pick.sort_values(by=g_pick.columns.tolist()).reset_index(drop=True)

        Q = len(morality_groups()[topic].keys())
        # ===== prompt concatenate =====
        for _, moral_row in g_pick.iterrows():
            for pct_q in pct_questions:
                lines = []
                lines.append("**Your Task**")
                lines.append(f"Your value orientation has been described through your answers to the following questions:")
                
                lines.append("")
                agree_list = []
                
                for idx in range(Q): 
                    opt_text = moral_row[f"a{idx+1}_text"]
                    
                    if "like" in opt_text.lower():
                        agree_list.append(idx)
                    
                
                if agree_list:
                    lines.append("In each case, make your choice in terms of how you feel right now on average, not what you have felt in the past or would like to feel. Reply with one of the following options:")
                    if MORAL_MODE == 2:
                        lines.append("   [0] Not like me")
                        lines.append("   [1] Like me")
                    else:
                        lines.append("   [0] Not at all like me")
                        lines.append("   [1] Not like me")
                        lines.append("   [2] A little like me")
                        lines.append("   [3] Somewhat like me")
                        lines.append("   [4] Like me")
                        lines.append("   [5] Very much like me")

                    lines.append("")
                    count=0
                    for idx in agree_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_agree[opt_id]
                            
                        lines.append(f"Case {count+1}: {q_text} -> Your answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                
                lines.append("")
                lines.append(f"Based on the value orientation your answers imply above, answer the question:")
                lines.append(f"{pct_q}")
                
                lines.append("")
                lines.append("**Instructions**")
                lines.append("Respond in *exactly two short parts*:")
                lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
                lines.append("2. *Brief Reason* – *one sentence* that clearly states your reasoning, consistent with the value orientation expressed above.")

                lines.append("")
                lines.append("**Important**")
                lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
                lines.append("- *Do NOT add any extra content.*")
                             
                lines.append("")
                lines.append("**Output Format**")
                lines.append("```json")
                lines.append("{")
                lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
                lines.append("  \"Brief Reason\": \"<one sentence>\"")
                lines.append("}")
                lines.append("```")
            
                prompt_text = "\n".join(lines)

                '''
                print(prompt_text)
                sys.exit(0)
                '''
                
                row_out = {
                    "topic": topic,
                    "score": score,
                    "question ids": moral_row.get("question id", ""),
                    "pct_question": pct_q,
                    "prompt": prompt_text
                }
                for idx in range(Q):
                    row_out[f"question {idx+1}"] = moral_row[f"question {idx+1}"]
                    row_out[f"answer {idx+1} id"] = int(moral_row[f"a{idx+1}_id"])
                    row_out[f"answer {idx+1} text"] = moral_row[f"a{idx+1}_text"]
                out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["topic", "score", "pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    print("Rows of each topic：")
    print(df_out['topic'].value_counts().sort_index().to_string())
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()
